## NBA 3.1 | PX Patient Features

Builds First Occurrence, Recency, Frequency, and Hospitalization features for PX claims.

**Input files:**
- All features (First Occ, Recency, Frequency): `px_all` (673M rows, 2021-01-02→2026-08-31)
  - Note: `px_full_history` was removed from INPUT_PATHS_31 — it was an identical duplicate of `px_all`. Use `px_all` only.
- Hospitalization: `px_hosp` (pre-filtered, 315M rows, 2021-01-02→2026-08-31)

**Output:** `.../kerendia_nba_3_1_volume/notebook_output/patient_features/px/`

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_1_develop_sa/NBA 3.1 - Dev/02_Input_Data_Creation/00_event_list_config"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_1_develop_sa/NBA 3.1 - Dev/00_Helper_Notebooks/00_cohort_generation"

In [0]:
import re
from functools import reduce
from pyspark.sql import functions as F, Window

FEATURE_FAMILY  = "claims"
CLAIM_TYPE      = "px"
EVENT_TYPE_VAL  = "px"

# PX: single non-hosp file (no windowed variant exists)
PX_ALL_PATH  = INPUT_PATHS_31["px_all"]   # First Occ + Recency + Frequency
PX_HOSP_PATH = INPUT_PATHS_31["px_hosp"]  # Hospitalization (pre-filtered)

OUTPUT_BASE  = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/"
                "kerendia_nba_3/kerendia_nba_3_1_volume/notebook_output/patient_features/px")
FO_OUT   = f"{OUTPUT_BASE}/px_first_occurrence_features"
REC_OUT  = f"{OUTPUT_BASE}/px_recency_features"
FREQ_OUT = f"{OUTPUT_BASE}/px_frequency_features"
HOSP_OUT = f"{OUTPUT_BASE}/px_hospitalization_features"

dbutils.widgets.text("TEST_HCP_SAMPLE", "0", "HCP sample size (0=full)")
TEST_HCP_SAMPLE  = int(dbutils.widgets.get("TEST_HCP_SAMPLE"))
RECENCY_WINDOWS  = [30, 60, 90, 180, 270, 360]
LAG_WINDOWS      = [1, 2, 3, 6, 12]
CHANGE_WINDOWS   = [3, 6]
PATIENT_WINDOWS  = [30, 60, 90, 180, 360]
HOSP_LAG_WINDOWS = [1, 6, 12]
HOSP_PAT_WINDOWS = [30, 180, 360]
EVENT_COLLAPSED  = "ALL_PROCEDURES"

print(f"PX_ALL_PATH  : {PX_ALL_PATH}")
print(f"PX_HOSP_PATH : {PX_HOSP_PATH}")
print(f"OUTPUT_BASE  : {OUTPUT_BASE}")

In [0]:
cohorts = CohortGenerator(FEATURE_FAMILY).load()
meta = cohorts.attrs["metadata"]
assert FEATURE_FAMILY == "claims" and meta["feature_family"] == "claims" and meta["offset_days"] == 0
assert list(cohorts["cohort_id"]) == [f"c{i}" for i in range(1, 29)]
c1 = cohorts.iloc[0]
assert (str(c1["lookback_start_date"]), str(c1["lookback_end_date"])) == ("2023-06-01", "2024-05-01")
print(f"Cohorts loaded: {len(cohorts)} rows")

In [0]:
def sanitize_column_name(name):
    name = re.sub(r'[^A-Za-z0-9_]', '_', name.replace("+", "AND").replace("/", "_OR_"))
    name = re.sub(r'_+', '_', name).strip('_')
    return name.upper()

def standardize(df, event_type):
    COLUMN_MAP = {
        "PROVIDER_ID": ["PROVIDER_ID", "HCP_ID"],
        "PATIENT_ID":  ["PATIENT_ID", "MEMBER_ID"],
        "EVENT_NAME":  ["EVENT_NAME", "PROCEDURE_DESCRIPTION", "PRODUCT_GROUP",
                        "DIAGNOSIS_DESCRIPTION", "TEST_TYPE"],
        "EVENT_DATE":  ["SVC_DT", "SERVICE_DATE"],
        "CLAIM_ID":    ["CLAIM_ID", "CLAIM_NBR"],
    }
    for canonical, variants in COLUMN_MAP.items():
        for v in variants:
            if v in df.columns and canonical not in df.columns:
                df = df.withColumnRenamed(v, canonical)
                break
    df = df.withColumn("EVENT_TYPE", F.lit(event_type))
    keep = [c for c in ["PROVIDER_ID", "PATIENT_ID", "EVENT_NAME", "EVENT_DATE",
                        "EVENT_TYPE", "CLAIM_ID"] if c in df.columns]
    return df.select(*keep)

In [0]:
HCP_UNIVERSE_PATH = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/"
                     "kerendia_nba_3/kerendia_nba_3_1_volume/HCP_Universe")
hcp_universe_df = spark.read.parquet(HCP_UNIVERSE_PATH).select("PROVIDER_ID").distinct()
if TEST_HCP_SAMPLE > 0:
    hcp_universe_df = (hcp_universe_df.orderBy(F.xxhash64("PROVIDER_ID"), "PROVIDER_ID")
                                      .limit(TEST_HCP_SAMPLE))
hcp_universe = F.broadcast(hcp_universe_df)
px_event_keys = event_groups("px")
print(f"Universe HCPs: {'TEST:'+str(TEST_HCP_SAMPLE) if TEST_HCP_SAMPLE > 0 else '314,920 (full)'}")
print(f"Px EVENT_GROUPs: {len(px_event_keys)}")

In [0]:
claims_all = (spark.read.format("delta").load(PX_ALL_PATH)
               .join(hcp_universe, "PROVIDER_ID", "left_semi"))
claims_all = standardize(claims_all, EVENT_TYPE_VAL)
validate_event_input(claims_all, CLAIM_TYPE)
claims_freq = (claims_all
               .withColumn("SVC_DT_M", F.trunc(F.col("EVENT_DATE"), "month"))
               .select("PROVIDER_ID", "PATIENT_ID", "EVENT_NAME", "SVC_DT_M",
                       "EVENT_DATE", "CLAIM_ID"))
print("px_all loaded and standardized")

In [0]:
fo_cols_raw = [sanitize_column_name(f"MAX_EVENT_TIME_{e}_FIRST_OCCURRENCE") for e in px_event_keys]
pre  = (claims_all.groupBy("PROVIDER_ID", "EVENT_NAME")
                  .agg(F.min("EVENT_DATE").alias("FIRST_OCCURRENCE_DATE")))
wide = (pre.groupBy("PROVIDER_ID")
           .pivot("EVENT_NAME", px_event_keys)
           .agg(F.min("FIRST_OCCURRENCE_DATE")))
for orig, canon in zip(px_event_keys, fo_cols_raw):
    safe = sanitize_column_name(orig)
    if safe in wide.columns:   wide = wide.withColumnRenamed(safe, canon)
    elif orig in wide.columns: wide = wide.withColumnRenamed(orig, canon)

cohort_anchor_pd = cohorts[["cohort_id", "lookback_end_date"]].copy()
cohort_anchor_pd.columns = ["COHORT", "LOOKBACK_END_DATE"]
fo = wide.crossJoin(F.broadcast(spark.createDataFrame(cohort_anchor_pd)))
for canon in [c for c in fo_cols_raw if c in fo.columns]:
    cutoff = F.last_day(F.col("LOOKBACK_END_DATE"))
    fo = fo.withColumn(canon, F.when(F.datediff(cutoff, F.col(canon)) >= 0,
                                     F.datediff(cutoff, F.col(canon))).otherwise(F.lit(None)))
fo = fo.filter(F.greatest(*[F.col(c).isNotNull().cast("int") for c in fo_cols_raw if c in fo.columns]) == 1)
fo = fo.withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT"))
(fo.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .partitionBy("COHORT").save(FO_OUT))
print(f"Written: {FO_OUT}")

In [0]:
rec_parts = []
for _, cohort_row in cohorts.iterrows():
    COHORT, LOOKBACK_START_DATE, LOOKBACK_END_DATE = (
        cohort_row["cohort_id"], cohort_row["lookback_start_date"], cohort_row["lookback_end_date"])
    last_day = F.last_day(F.lit(LOOKBACK_END_DATE))
    c_claims = (claims_all
                .filter((F.col("EVENT_DATE") >= F.lit(LOOKBACK_START_DATE)) &
                        (F.col("EVENT_DATE") < F.add_months(F.lit(LOOKBACK_END_DATE), 1)))
                .withColumn("EVENT_TIME", F.datediff(last_day, F.col("EVENT_DATE")))
                .withColumn("COHORT", F.lit(COHORT))
                .withColumn("HCP_COHORT_ID", F.concat_ws("_", F.col("PROVIDER_ID"), F.lit(COHORT))))
    sub1 = (c_claims.groupBy("HCP_COHORT_ID").pivot("EVENT_NAME", px_event_keys).agg(F.min("EVENT_TIME")))
    for e in px_event_keys:
        safe = sanitize_column_name(e)
        if safe in sub1.columns:   sub1 = sub1.withColumnRenamed(safe, f"MIN_EVENT_TIME_{safe}_TIME_MIN")
        elif e in sub1.columns:    sub1 = sub1.withColumnRenamed(e, f"MIN_EVENT_TIME_{sanitize_column_name(e)}_TIME_MIN")
    sub2 = (c_claims.groupBy("HCP_COHORT_ID")
                    .agg(F.count("CLAIM_ID").alias("TOTAL_EVENTS_PX"),
                         F.countDistinct("EVENT_NAME").alias("UNIQUE_EVENTS_PX")))
    sub3 = reduce(lambda a, b: a.join(b, "HCP_COHORT_ID", "outer"),
                  [c_claims.filter(F.col("EVENT_TIME") <= w).groupBy("HCP_COHORT_ID")
                           .agg(F.count("CLAIM_ID").alias(f"TOTAL_EVENTS_TIME_{w}_DAYS_PX"),
                                F.countDistinct("EVENT_NAME").alias(f"UNIQUE_EVENTS_TIME_{w}_DAYS_PX"))
                   for w in RECENCY_WINDOWS])
    sub6 = reduce(lambda a, b: a.join(b, "HCP_COHORT_ID", "outer"),
                  [c_claims.filter(F.col("EVENT_TIME") <= w).groupBy("HCP_COHORT_ID")
                           .agg(F.skewness("EVENT_TIME").alias(f"SKEWNESS_TIME_{w}_DAYS_PX"))
                   for w in RECENCY_WINDOWS])
    all_ids = c_claims.select("HCP_COHORT_ID", "PROVIDER_ID", "COHORT").distinct()
    rec_parts.append(all_ids.join(sub1, "HCP_COHORT_ID", "left")
                            .join(sub2, "HCP_COHORT_ID", "left")
                            .join(sub3, "HCP_COHORT_ID", "left")
                            .join(sub6, "HCP_COHORT_ID", "left"))
recency = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), rec_parts)
(recency.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .partitionBy("COHORT").save(REC_OUT))
print(f"Written: {REC_OUT}")

In [0]:
freq_a_cols = {e: f"FREQ_{sanitize_column_name(e)}" for e in px_event_keys}
freq_a = (claims_freq.groupBy("PROVIDER_ID", "SVC_DT_M")
                     .pivot("EVENT_NAME", px_event_keys).agg(F.count(F.lit(1))))
for e in px_event_keys:
    safe = sanitize_column_name(e)
    if safe in freq_a.columns:   freq_a = freq_a.withColumnRenamed(safe, freq_a_cols[e])
    elif e in freq_a.columns:    freq_a = freq_a.withColumnRenamed(e, freq_a_cols[e])
freq_cols = [c for c in freq_a.columns if c not in ("PROVIDER_ID", "SVC_DT_M")]
freq_a = freq_a.fillna(0, subset=freq_cols)

# Sub-B
min_m = claims_freq.agg(F.min("SVC_DT_M")).first()[0]
max_m = claims_freq.agg(F.max("SVC_DT_M")).first()[0]
months_range = (spark.range(0, 1000)
    .select(F.expr(f"date_add(date '{min_m}', cast(id * 30 as int))").alias("_d"))
    .filter(F.col("_d") <= F.lit(max_m))
    .select(F.trunc("_d", "month").alias("SVC_DT_M")).distinct())
spine = claims_freq.select("PROVIDER_ID").distinct().crossJoin(months_range)
freq_b = spine.join(freq_a, ["PROVIDER_ID", "SVC_DT_M"], "left").fillna(0, subset=freq_cols)
w_spec = Window.partitionBy("PROVIDER_ID").orderBy("SVC_DT_M")
# Single projection: keys + all lag columns (raw FREQ_* columns dropped implicitly)
lag_exprs = [F.sum(c).over(w_spec.rowsBetween(-(n-1), 0)).alias(f"{c}_IN_LAST_{n}_MONTH")
             for c in freq_cols for n in LAG_WINDOWS]
freq_b = freq_b.select("PROVIDER_ID", "SVC_DT_M", *lag_exprs)

# Sub-C: log-ratio
existing_cols = set(freq_b.columns)
logratio_exprs = []
for col_name in [f"FREQ_{sanitize_column_name(e)}" for e in px_event_keys]:
    f12 = f"{col_name}_IN_LAST_12_MONTH"
    if f12 not in existing_cols: continue
    for n in CHANGE_WINDOWS:
        fn = f"{col_name}_IN_LAST_{n}_MONTH"
        if fn not in existing_cols: continue
        logratio_exprs.append(
            F.log(((F.col(fn)/n)+0.5)/(((F.col(f12)-F.col(fn))/(12-n))+0.5))
             .alias(f"LOGRATIO_{col_name.replace('FREQ_','FREQ_')}_IN_LAST_{n}_MONTH"))
freq_b = freq_b.select("*", *logratio_exprs)

# Sub-D
pat_parts = []
for _, cohort_row in cohorts.iterrows():
    COHORT, LOOKBACK_START_DATE, LOOKBACK_END_DATE = (
        cohort_row["cohort_id"], cohort_row["lookback_start_date"], cohort_row["lookback_end_date"])
    last_day_expr = F.last_day(F.lit(LOOKBACK_END_DATE))
    c_pat = (claims_freq
             .filter((F.col("SVC_DT_M") >= F.lit(LOOKBACK_START_DATE)) &
                     (F.col("SVC_DT_M") < F.add_months(F.lit(LOOKBACK_END_DATE), 1)))
             .withColumn("DAYS_DIFF", F.datediff(last_day_expr, F.last_day(F.col("SVC_DT_M")))))
    sub_d_aggs = []
    for w in PATIENT_WINDOWS:
        pivoted = (c_pat.filter(F.col("DAYS_DIFF") <= w)
                        .dropDuplicates(["PROVIDER_ID", "EVENT_NAME", "PATIENT_ID"])
                        .groupBy("PROVIDER_ID", "EVENT_NAME").agg(F.countDistinct("PATIENT_ID").alias("_cnt"))
                        .groupBy("PROVIDER_ID").pivot("EVENT_NAME", px_event_keys).agg(F.max("_cnt")))
        for e in px_event_keys:
            safe = sanitize_column_name(e)
            new_col = f"NUM_OF_PATIENTS_THE_HCP_DID_PROCEDURE_{safe}_IN_LAST_{w}_DAYS"
            if safe in pivoted.columns:   pivoted = pivoted.withColumnRenamed(safe, new_col)
            elif e in pivoted.columns:    pivoted = pivoted.withColumnRenamed(e, new_col)
        sub_d_aggs.append(pivoted)
    pat_parts.append(reduce(lambda a, b: a.join(b, "PROVIDER_ID", "outer"), sub_d_aggs)
                    .withColumn("COHORT", F.lit(COHORT)))
freq_d = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), pat_parts)

cohort_range_pd = [{"COHORT": row["cohort_id"], "COHORT_MONTH_START": row["lookback_start_date"],
                    "COHORT_MONTH_END": row["lookback_end_date"]} for _, row in cohorts.iterrows()]
cohort_range_sp = spark.createDataFrame(cohort_range_pd)
freq_bc_cohort = (freq_b.crossJoin(F.broadcast(cohort_range_sp))
                        .filter((F.col("SVC_DT_M") >= F.col("COHORT_MONTH_START")) &
                                (F.col("SVC_DT_M") < F.add_months(F.col("COHORT_MONTH_END"), 1)))
                        .drop("COHORT_MONTH_START", "COHORT_MONTH_END"))
frequency = freq_bc_cohort.join(freq_d, ["PROVIDER_ID", "COHORT"], "left")
(frequency.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
          .partitionBy("COHORT").save(FREQ_OUT))
print(f"Written: {FREQ_OUT}")

In [0]:
# Load pre-filtered PX hosp file — no DATA_SOURCE filter needed
hosp_claims = (spark.read.format("delta").load(PX_HOSP_PATH)
                .join(hcp_universe, "PROVIDER_ID", "left_semi"))
hosp_claims = standardize(hosp_claims, EVENT_TYPE_VAL)
hosp_claims = (hosp_claims
               .withColumn("SVC_DT_M", F.trunc(F.col("EVENT_DATE"), "month"))
               .withColumn("EVENT_NAME", F.lit(EVENT_COLLAPSED))
               .select("PROVIDER_ID", "PATIENT_ID", "SVC_DT_M", "EVENT_NAME", "CLAIM_ID"))

total_col = f"TOTAL_CLAIMS_{EVENT_COLLAPSED}"
hosp_col  = f"TOTAL_HOSPITALIZED_CLAIMS_{EVENT_COLLAPSED}"

total_monthly = (claims_freq.withColumn("EVENT_NAME", F.lit(EVENT_COLLAPSED))
                             .groupBy("PROVIDER_ID", "SVC_DT_M")
                             .pivot("EVENT_NAME", [EVENT_COLLAPSED])
                             .agg(F.count(F.lit(1)))
                             .withColumnRenamed(EVENT_COLLAPSED, total_col))
hosp_counts   = (hosp_claims.groupBy("PROVIDER_ID", "SVC_DT_M")
                             .pivot("EVENT_NAME", [EVENT_COLLAPSED])
                             .agg(F.count(F.lit(1)))
                             .withColumnRenamed(EVENT_COLLAPSED, hosp_col))
hosp_a = (total_monthly.join(hosp_counts, ["PROVIDER_ID", "SVC_DT_M"], "left")
                        .fillna(0, subset=[hosp_col]))
print(f"Hosp Sub-A columns: {hosp_a.columns}")

In [0]:
min_m_h = claims_freq.agg(F.min("SVC_DT_M")).first()[0]
max_m_h = claims_freq.agg(F.max("SVC_DT_M")).first()[0]
months_h = (spark.range(0, 1000)
    .select(F.expr(f"date_add(date '{min_m_h}', cast(id * 30 as int))").alias("_d"))
    .filter(F.col("_d") <= F.lit(max_m_h))
    .select(F.trunc("_d", "month").alias("SVC_DT_M")).distinct())
spine_h = hosp_a.select("PROVIDER_ID").distinct().crossJoin(months_h)
hosp_filled = (spine_h.join(hosp_a, ["PROVIDER_ID", "SVC_DT_M"], "left")
                      .fillna(0, subset=[total_col, hosp_col]))
w_h = Window.partitionBy("PROVIDER_ID").orderBy("SVC_DT_M")
for c in [total_col, hosp_col]:
    for n in HOSP_LAG_WINDOWS:
        hosp_filled = hosp_filled.withColumn(f"{c}_IN_LAST_{n}_MONTH",
                                             F.sum(c).over(w_h.rowsBetween(-(n-1), 0)))
hosp_filled = hosp_filled.drop(total_col, hosp_col)

# Validation filter
filter_cond = F.lit(True)
for n in HOSP_LAG_WINDOWS:
    filter_cond = filter_cond & (
        F.col(f"{total_col}_IN_LAST_{n}_MONTH") >= F.col(f"{hosp_col}_IN_LAST_{n}_MONTH"))
hosp_b = hosp_filled.filter(filter_cond)

# Sub-C: percent hospitalized
hosp_c = hosp_b
for n in HOSP_LAG_WINDOWS:
    tc = f"{total_col}_IN_LAST_{n}_MONTH"
    hc = f"{hosp_col}_IN_LAST_{n}_MONTH"
    hosp_c = hosp_c.withColumn(
        f"PERC_HOSPITALIZED_HCP_CLAIMS_FOR_{EVENT_COLLAPSED}_IN_LAST_{n}_MONTH",
        F.when(F.col(tc) != 0, F.col(hc)/F.col(tc)).otherwise(F.lit(None)))
hosp_c = hosp_c.drop(*[f"{total_col}_IN_LAST_{n}_MONTH" for n in HOSP_LAG_WINDOWS])

# Sub-D: unique hospitalized patients per cohort
hosp_pat_parts = []
for _, cohort_row in cohorts.iterrows():
    COHORT, LOOKBACK_START_DATE, LOOKBACK_END_DATE = (
        cohort_row["cohort_id"], cohort_row["lookback_start_date"], cohort_row["lookback_end_date"])
    last_day_expr = F.last_day(F.lit(LOOKBACK_END_DATE))
    c_hosp = (hosp_claims
              .filter((F.col("SVC_DT_M") >= F.lit(LOOKBACK_START_DATE)) &
                      (F.col("SVC_DT_M") < F.add_months(F.lit(LOOKBACK_END_DATE), 1)))
              .withColumn("DAYS_DIFF", F.datediff(last_day_expr, F.last_day(F.col("SVC_DT_M")))))
    sub_d_aggs = [
        c_hosp.filter(F.col("DAYS_DIFF") <= w).dropDuplicates(["PROVIDER_ID", "PATIENT_ID"])
              .groupBy("PROVIDER_ID").agg(F.countDistinct("PATIENT_ID").alias(
                  f"NUM_OF_HOSPITALIZED_PATIENTS_THE_HCP_DID_PROCEDURE_IN_LAST_{w}_DAYS"))
        for w in HOSP_PAT_WINDOWS]
    hosp_pat_parts.append(
        reduce(lambda a, b: a.join(b, "PROVIDER_ID", "outer"), sub_d_aggs)
        .withColumn("COHORT", F.lit(COHORT)))
hosp_d = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), hosp_pat_parts)

# Attach cohort to monthly grain + join Sub-D
hosp_bc_cohort = (hosp_c.crossJoin(F.broadcast(cohort_range_sp))
                        .filter((F.col("SVC_DT_M") >= F.col("COHORT_MONTH_START")) &
                                (F.col("SVC_DT_M") < F.add_months(F.col("COHORT_MONTH_END"), 1)))
                        .drop("COHORT_MONTH_START", "COHORT_MONTH_END"))
hosp_final = hosp_bc_cohort.join(hosp_d, ["PROVIDER_ID", "COHORT"], "left")
(hosp_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
           .partitionBy("COHORT").save(HOSP_OUT))
print(f"Written: {HOSP_OUT}")

In [0]:
for path, label in [(FO_OUT, "First Occurrence"), (REC_OUT, "Recency"),
                    (FREQ_OUT, "Frequency"), (HOSP_OUT, "Hospitalization")]:
    df = spark.read.format("delta").load(path)
    cohort_ids = {r[0] for r in df.select("COHORT").distinct().collect()}
    expected   = set(cohorts["cohort_id"])
    n_outside  = (df.select("PROVIDER_ID").distinct()
                    .join(hcp_universe_df, "PROVIDER_ID", "left_anti").count())
    print(f"{label}: {df.count():,} rows | cohorts={len(cohort_ids)}/28 "
          f"| missing={expected-cohort_ids or 'none'} | outside universe={n_outside}")
dbutils.notebook.exit("PX patient features written: First Occurrence, Recency, Frequency, Hospitalization")